In [ ]:
import pygame
import serial
import time
import struct
import os

# --- CONFIGURATION ---
SERIAL_PORT = 'COM3'  # Update if necessary
BAUD_RATE = 115200
UPDATE_HZ = 50       # How often to send commands (Hz)
DEADZONE = 0.05

def main():
    # Initialize Serial
    serial = None
    takeover = False
    
    try:
        serial = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
        time.sleep(2) 
        print(f"Connected to {SERIAL_PORT}")
    except Exception as e:
        print(f"Warning: Could not open {SERIAL_PORT}: {e}")

    # Initialize Pygame for Controller
    pygame.init()
    pygame.joystick.init()

    if pygame.joystick.get_count() == 0:
        print("No Xbox controller detected; Closing arduino and returning!")
        if serial: serial.close()
        return

    controller = pygame.joystick.Joystick(0)
    controller.init()
    print(f"Connected to: {controller.get_name()}")

    running = True
    clock = pygame.time.Clock()


    try:
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

            pygame.event.pump()
            
            # Read Axes
            raw_lt = controller.get_axis(4) # Brake
            raw_rt = controller.get_axis(5) # Throttle
            raw_st = controller.get_axis(2) # Steering (Right Stick X)
            raw_ac = controller.get_axis(3) # Actuator (Right Stick Y)
            recover_takeover = controller.get_button(3) # LB to recover takeover
            
            if recover_takeover:
                takeover = False
                print(f"\nUser has recovered control of the vehicle, resuming command transmission to arduino.")
                
            regen_brake = 0
            throttle = 0
            actuator = 0
            
            # 1. THROTTLE/BRAKE: Map [0, 1] to [0, 255]
            if raw_lt > DEADZONE:
                regen_brake = int(raw_lt * 255)
            
            if raw_rt > DEADZONE:
                throttle = int(raw_rt * 255)
            
            if raw_ac > DEADZONE: # Small deadzone
                actuator = int(raw_ac * 255)

            # 2. STEERING: Map [-1, 1] to [0, 255], 127 = center
            steering = int(((raw_st + 1) / 2) * 255)
            
            

            # Data Packet: [StartByte, Throttle, RegenBrake, Steering, Actuator]
            start_byte = 0xAA
            checksum = (start_byte + throttle + regen_brake + steering + actuator) & 0xFF
            packet = struct.pack('BBBBBB', start_byte, throttle, regen_brake, steering, actuator, checksum)

            if serial:
                try:
                    if not takeover:
                        serial.write(packet) # writing to serial port
                        print(f"sending commands to arduino: T:{throttle} R:{regen_brake} S:{steering} A:{actuator}")
                    else:
                        print(f"vehicle is safely taken over by user, not sending commands to arduino")
                    
                    if serial.in_waiting:
                        line = serial.readline().decode('utf-8', errors='ignore').strip()
                        if line.startswith("FB"):
                            try:
                                # Expected: FB T:0 R:0 S:127 A:0 AS:0.0 RPM:0 M:0
                                parts = line.split()
                                data = {p.split(':')[0]: p.split(':')[1] for p in parts if ':' in p}
                                
                                # Conversion to Units
                                t_perc = (int(data.get('T', 0)) / 255.0) * 100
                                r_perc = (int(data.get('R', 0)) / 255.0) * 100
                                a_perc = (int(data.get('A', 0)) / 255.0) * 100
                                s_raw  = int(data.get('S', 127))
                                s_deg  = (s_raw - 127) * (540.0 / 127.0)
                                
                                as_deg = float(data.get('AS', 0.0))
                                rpm    = int(data.get('RPM', 0))
                                status = "OK" if data.get('M') == "1" else "NO_MTR"
                                
                                if t_perc > 20 or r_perc > 20:
                                    takeover = True
                                    print(f"\nTakeover triggered by user input!")

                                # Build comprehensive display line
                                display = (
                                    f"CMD: T:{t_perc:3.0f}% R:{r_perc:3.0f}% S:{s_deg:4.0f}° A:{a_perc:3.0f}% | "
                                    f"FB: AS:{as_deg:5.1f}° RPM:{rpm:4d} | {status}"
                                )
                                print(f"\r[{time.strftime('%H:%M:%S')}] {display:80}", end='')
                            except Exception as e:
                                print(f"\r[{time.strftime('%H:%M:%S')}] FB: {line:80}", end='')
                                raise e
                        else:
                            print(line)

                except serial.SerialTimeoutException:
                    print(f"\r[{time.strftime('%H:%M:%S')}] ERROR: Serial Write Timeout! Is Arduino frozen?", end='')
                    raise e 
                
                except Exception as e:
                    print(f"\r[{time.strftime('%H:%M:%S')}] Error: {e}", end='')
            else:
                print(f"serial port not available, cannot send commands to arduino")
                
            clock.tick(UPDATE_HZ)

    except KeyboardInterrupt:
        print("\nStopping...")
    finally:
        if serial:
            # Send neutral/zero on exit
            packet = struct.pack('BBBBBB', 0xAA, 0, 0, 127, 0, (0xAA + 0 + 0 + 127 + 0) & 0xFF)
            serial.write(packet)
            serial.close()
        pygame.quit()

if __name__ == "__main__":
    main()



#include <SPI.h>
#include <mcp_can.h>
#include "pwm.h" // Native R4 PWM library if needed, or analogWrite
#include <Arduino_CAN.h> // Native CAN for Curtis Feedback

// --- HARDWARE PINS ---
const int throttlePin = 9;      // Curtis Throttle
const int regenBrakePin = 3;    // Curtis Regen Brake (Moved to avoid SPI CS)
const int actuatorPinForward = 5; 
const int actuatorPinReverse = 6; 
const int pedalPin = A0;      

// --- MCP2515 CONFIG ---
const int SPI_CS_PIN = 10;
MCP_CAN CAN0(SPI_CS_PIN);

// --- steering motor CONFIG (VESC Protocol) ---
const uint32_t MOTOR_ID = 104;  
const float MAX_STEER_DEG = 540.0; // 1.5 Rotations (540 degrees)

// --- TUNING PARAMETERS ---
float steer_kp = 0.5; 
// To make steering "faster" or "slower" from Arduino:
float current_pos = 0;
float max_degrees_per_loop = 1.5; // Lowered to 5.0 for smoother feel (500 deg/sec)

// --- SAFETY CONFIG ---
const unsigned long SERIAL_TIMEOUT_MS = 1000;
unsigned long lastPacketTime = 0;
bool motorHeard = false;

// --- STATE ---
float current_curtis_rpm = 0;
float current_steer_angle = 0;

// Current values from Xbox
int xboxThrottle = 0;
int xboxRegenBrake = 0;
int xboxSteering = 127; 
int xboxActuator = 0;

void setup() {
    Serial.begin(115200);
    
    unsigned long start = millis();

    while (!Serial) {
      delay(50); // wait
    }
    Serial.println("Arduino is ready.");

    delay(1000);

    pinMode(LED_BUILTIN, OUTPUT);
    digitalWrite(LED_BUILTIN, HIGH); 

    analogWriteResolution(8);  // this is default 
    
    pinMode(throttlePin, OUTPUT);
    pinMode(regenBrakePin, OUTPUT);
    pinMode(actuatorPinForward, OUTPUT);
    pinMode(actuatorPinReverse, OUTPUT);
    
    while (CAN0.begin(MCP_ANY, CAN_1000KBPS, MCP_8MHZ) != CAN_OK) {
        Serial.println("CAN Init Failed");
        
        delay(250);
    }
    Serial.println("Steering CAN Init OK!");

    CAN0.setMode(MCP_NORMAL);
    lastPacketTime = millis();

    while (!CAN.begin(CanBitRate::BR_250k)){
        Serial.println("Native CAN Fail");
    }
    Serial.println("Vehicle CAN Init OK!");
}

void loop() {
    // 1. Process Steering Feedback (MCP2515)
    if (CAN0.checkReceive() == CAN_MSGAVAIL) {
        long unsigned int rxId;
        unsigned char len = 0;
        unsigned char rxBuf[8];
        CAN0.readMsgBuf(&rxId, &len, rxBuf);
        
        uint32_t clean_id = rxId & 0x1FFFFFFF;

        // if (((clean_id >> 8) & 0xFF) == 0x10 && (clean_id & 0xFF) == MOTOR_ID) 
        if ((clean_id & 0xFF) == MOTOR_ID) {
            current_steer_angle = (int16_t)((rxBuf[6] << 8) | rxBuf[7]) / 10.0f;
            motorHeard = true;
        }
    }

    // 2. Process Curtis Feedback (Native CAN)
    while (CAN.available()) {
        CanMsg msg = CAN.read();
        if (msg.id == 0x601) { // Standard Curtis RPM ID
            current_curtis_rpm = (int16_t)((msg.data[5] << 8) | msg.data[4]);
        }
    }

    // 3. Process Xbox Serial Packet
    if (Serial.available() >= 6) {
        if (Serial.read() == 0xAA) {
            int t = Serial.read();
            int r = Serial.read();
            int s = Serial.read();
            int a = Serial.read();
            int checksum = Serial.read();
            
            if (((0xAA + t + r + s + a) & 0xFF) == checksum) {
                xboxThrottle = t;
                xboxRegenBrake = r;
                xboxSteering = s;
                xboxActuator = a;
                lastPacketTime = millis();
                
                // Send comprehensive FB string
                Serial.print("FB ");
                Serial.print("T:");   Serial.print(xboxThrottle);
                Serial.print(" R:");   Serial.print(xboxRegenBrake);
                Serial.print(" S:");   Serial.print(xboxSteering);
                Serial.print(" A:");   Serial.print(xboxActuator);
                Serial.print(" AS:");  Serial.print(current_steer_angle, 1);
                Serial.print(" RPM:"); Serial.print(current_curtis_rpm, 0);
                Serial.print(" M:");   Serial.println(motorHeard ? "1" : "0");
            }
        }
    }

    // 4. Safety/Timeout
    if (millis() - lastPacketTime > SERIAL_TIMEOUT_MS) {
        xboxThrottle = 0; xboxRegenBrake = 0; xboxSteering = 127; xboxActuator = 0;
    }

    // 4. Update Traction
    analogWrite(throttlePin, max(xboxThrottle, (int)map(analogRead(pedalPin), 0, 1023, 0, 255)));
    analogWrite(regenBrakePin, xboxRegenBrake);
    
    // 5. Update Steering (Position Control with Slew Rate)
    // Map stick (0..127..255) to (-540..0..540)
    float target_pos = (float)(xboxSteering - 127) * (MAX_STEER_DEG / 127.0f);
    
    // Slew Rate Filter: Limits how fast the motor can turn (Degrees per 10ms)
    float diff = target_pos - current_pos;
    if (diff > max_degrees_per_loop) diff = max_degrees_per_loop;
    if (diff < -max_degrees_per_loop) diff = -max_degrees_per_loop;
    current_pos += diff;
    
    send_pos_command(MOTOR_ID, current_pos);

    // 6. Update Actuator
    analogWrite(actuatorPinForward, xboxActuator);
    analogWrite(actuatorPinReverse, 0);

    delay(10); 
}

void send_pos_command(uint32_t controller_id, float pos) {
    // Scaling per documentation: pos * 10000.0
    int32_t p = (int32_t)(pos * 10000.0f); 
    
    byte buffer[4];
    // Big Endian as per docs (Data[0] is most significant)
    buffer[0] = (p >> 24) & 0xFF;
    buffer[1] = (p >> 16) & 0xFF;
    buffer[2] = (p >> 8)  & 0xFF;
    buffer[3] = p & 0xFF;
    
    uint32_t can_id = controller_id | ((uint32_t)0x04 << 8); 
    CAN0.sendMsgBuf(can_id, 1, 4, buffer);
}

byte send_rpm_command(uint32_t controller_id, long rpm) {
    byte buffer[4];
    buffer[0] = (rpm >> 24) & 0xFF;
    buffer[1] = (rpm >> 16) & 0xFF;
    buffer[2] = (rpm >> 8)  & 0xFF;
    buffer[3] = rpm & 0xFF;
    uint32_t can_id = controller_id | ((uint32_t)0x03 << 8);
    return CAN0.sendMsgBuf(can_id, 1, 4, buffer);
}